# Cleanup

In this notebook you delete the resources you created during the workshop, so that they no longer incur charges.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [Delete the Knowledge Base and data source](#s1)
4. [Delete the guardrail](#s2)
5. [Reset the Lambda function](#s3)
6. [Delete the S3 objects](#s4)
7. [Turn off invocation logging](#s5)
8. [Try it](#try-it)
9. [Conclusion and key takeaways](#conclusion)
10. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

Over the workshop you built AnyCompany's travel assistant one module at a time, and created a few AWS resources along the way. Some of them incur charges while they exist, so delete them when you finish:

| Resource | Created in | What this notebook does |
|:--|:--|:--|
| Managed Knowledge Base `aim317-travel-policy` and its data source | Module 3 | Deletes them, along with the indexed passages |
| Guardrail `aim317-travel-guardrail` | Module 4 | Deletes it and all its versions |
| Lambda function `aim317-travel-assistant` | Module 6 | Resets it to an empty shell at AWS events, or deletes it in your own account |
| S3 bucket with the policy documents | Module 3 | Deletes the documents and the bucket |
| Model invocation logging | Module 0 | Turns it off |

Each cell checks whether the resource exists first, so you can run this notebook more than once, and it is safe to run even if you skipped a module.

> **Note:** If you deployed the agent in the [AgentCore stretch notebook](stretch_agentcore.ipynb), run the last cell of that notebook to remove it. This notebook does not touch it.

## 2. Setup <a id="setup"></a>

Run the cell below to create the clients and set the names of the workshop resources. These are the same names the earlier modules used.

In [ ]:
import time

import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

s3 = boto3.client("s3", region_name=REGION)
iam = boto3.client("iam")
logs = boto3.client("logs", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)  # Guardrails and logging configuration
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)  # Knowledge bases

# Names used in the earlier modules.
KNOWLEDGE_BASE_NAME = "aim317-travel-policy"  # Module 3
BUCKET_NAME = f"aim317-travel-policy-{ACCOUNT_ID}-{REGION}"  # Module 3
GUARDRAIL_NAME = "aim317-travel-guardrail"  # Module 4
FUNCTION_NAME = "aim317-travel-assistant"  # Module 6
LOG_GROUP_NAME = "/aim317/bedrock-invocations"  # Module 0

print(f"Region:  {REGION}")
print(f"Account: {ACCOUNT_ID}")

## 3. Delete the Knowledge Base and data source <a id="s1"></a>

A knowledge base is deleted in two steps: first its data sources, then the knowledge base itself. In Module 3 you created the data source with `dataDeletionPolicy` set to `DELETE`, so deleting it also removes the passages it indexed.

Run the cell below to delete the knowledge base you created in Module 3. It leaves `aim317-travel-policy-prebuilt` alone: at AWS events, that fallback knowledge base belongs to the event account, not to you.

In [ ]:
def find_knowledge_base(name):
    """Return the ID of the knowledge base with this name, or None."""
    for kb in bedrock_agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"]:
        if kb["name"] == name:
            return kb["knowledgeBaseId"]
    return None


knowledge_base_id = find_knowledge_base(KNOWLEDGE_BASE_NAME)

if knowledge_base_id is None:
    print(f"Knowledge base {KNOWLEDGE_BASE_NAME} not found; nothing to delete")
else:
    for data_source in bedrock_agent.list_data_sources(knowledgeBaseId=knowledge_base_id)["dataSourceSummaries"]:
        bedrock_agent.delete_data_source(knowledgeBaseId=knowledge_base_id, dataSourceId=data_source["dataSourceId"])
        print(f"Deleting data source {data_source['dataSourceId']}")
    # Wait for the data sources to finish deleting before deleting the knowledge base.
    while bedrock_agent.list_data_sources(knowledgeBaseId=knowledge_base_id)["dataSourceSummaries"]:
        time.sleep(5)

    bedrock_agent.delete_knowledge_base(knowledgeBaseId=knowledge_base_id)
    while find_knowledge_base(KNOWLEDGE_BASE_NAME):
        time.sleep(5)
    print(f"Deleted knowledge base {KNOWLEDGE_BASE_NAME} ({knowledge_base_id})")

## 4. Delete the guardrail <a id="s2"></a>

A guardrail has a working draft and numbered versions. Deleting it without a version number deletes the draft and every version at once.

Run the cell below to delete the guardrail you created in Module 4.

In [ ]:
guardrail_id = next(
    (g["id"] for g in bedrock.list_guardrails(maxResults=100)["guardrails"] if g["name"] == GUARDRAIL_NAME),
    None,
)

if guardrail_id is None:
    print(f"Guardrail {GUARDRAIL_NAME} not found; nothing to delete")
else:
    bedrock.delete_guardrail(guardrailIdentifier=guardrail_id)
    print(f"Deleted guardrail {GUARDRAIL_NAME} ({guardrail_id}) and all its versions")

## 5. Reset the Lambda function <a id="s3"></a>

What happens to the Lambda function depends on where it came from:

- **At AWS events**, CloudFormation created the function before the workshop, and you updated its code in Module 6. The cell below puts back a minimal handler, so the function no longer calls any models, and removes the knowledge base and guardrail settings. CloudFormation deletes the function itself when the event ends.
- **In your own account**, Module 6 created the function, so the cell below deletes it.

The cell tells the two apart by the `aws:cloudformation:stack-name` tag, which CloudFormation adds to every function it creates.

In [ ]:
import io
import zipfile

SHELL_CODE = '''def lambda_handler(event, context):
    return {"error": "The travel assistant is not deployed. Run Module 6 to deploy it."}
'''

try:
    function_arn = lambda_client.get_function(FunctionName=FUNCTION_NAME)["Configuration"]["FunctionArn"]
except lambda_client.exceptions.ResourceNotFoundException:
    function_arn = None
    print(f"Lambda function {FUNCTION_NAME} not found; nothing to reset")

if function_arn:
    tags = lambda_client.list_tags(Resource=function_arn)["Tags"]
    if "aws:cloudformation:stack-name" in tags:
        package = io.BytesIO()
        with zipfile.ZipFile(package, "w") as archive:
            archive.writestr("lambda_function.py", SHELL_CODE)
        lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=package.getvalue())
        lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)
        lambda_client.update_function_configuration(
            FunctionName=FUNCTION_NAME, Handler="lambda_function.lambda_handler", Environment={"Variables": {}}
        )
        print(f"Reset {FUNCTION_NAME} to an empty shell (created by stack {tags['aws:cloudformation:stack-name']})")
    else:
        lambda_client.delete_function(FunctionName=FUNCTION_NAME)
        print(f"Deleted Lambda function {FUNCTION_NAME}")

## 6. Delete the S3 objects <a id="s4"></a>

The knowledge base read AnyCompany's policy documents from an S3 bucket that you created in Module 3. A bucket must be empty before you can delete it.

Run the cell below to delete the documents and then the bucket.

In [ ]:
try:
    s3.head_bucket(Bucket=BUCKET_NAME)
except s3.exceptions.ClientError:
    print(f"Bucket {BUCKET_NAME} not found; nothing to delete")
else:
    deleted = 0
    for page in s3.get_paginator("list_object_versions").paginate(Bucket=BUCKET_NAME):
        objects = [{"Key": v["Key"], "VersionId": v["VersionId"]}
                   for v in page.get("Versions", []) + page.get("DeleteMarkers", [])]
        if objects:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": objects})
            deleted += len(objects)
    s3.delete_bucket(Bucket=BUCKET_NAME)
    print(f"Deleted {deleted} objects and the bucket {BUCKET_NAME}")

## 7. Turn off invocation logging <a id="s5"></a>

Model invocation logging is a setting for the whole account and Region, so it keeps recording every model call until you turn it off. The cell below turns it off only if it still points at the workshop log group, so it does not undo a logging setup that someone else configured later.

The records already in `/aim317/bedrock-invocations` expire after the 7 days you set in Module 0.

In [ ]:
config = bedrock.get_model_invocation_logging_configuration().get("loggingConfig", {})
log_group = config.get("cloudWatchConfig", {}).get("logGroupName")

if log_group == LOG_GROUP_NAME:
    bedrock.delete_model_invocation_logging_configuration()
    print("Model invocation logging is off")
elif log_group:
    print(f"Logging points at {log_group}, not the workshop log group, so it was left on")
else:
    print("Model invocation logging is already off")

## 8. Try it <a id="try-it"></a>

1. Run the cell below to confirm that none of the workshop resources are left. Every line should say `gone`.
2. If you ran the workshop in your own account rather than at an AWS event, the notebooks also created three IAM roles and a log group. They cost nothing, but you can remove them too: set `DELETE_ROLES_AND_LOGS = True` in the cell after next and run it.

In [ ]:
def status(found):
    return "still exists" if found else "gone"


try:
    lambda_client.get_function(FunctionName=FUNCTION_NAME)
    function_found = True
except lambda_client.exceptions.ResourceNotFoundException:
    function_found = False

try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    bucket_found = True
except s3.exceptions.ClientError:
    bucket_found = False

guardrail_found = any(g["name"] == GUARDRAIL_NAME for g in bedrock.list_guardrails(maxResults=100)["guardrails"])
logging_found = (bedrock.get_model_invocation_logging_configuration().get("loggingConfig", {})
                 .get("cloudWatchConfig", {}).get("logGroupName") == LOG_GROUP_NAME)

print(f"Knowledge base:      {status(find_knowledge_base(KNOWLEDGE_BASE_NAME))}")
print(f"Guardrail:           {status(guardrail_found)}")
print(f"Lambda function:     {status(function_found)}  (at AWS events it stays, as an empty shell)")
print(f"S3 bucket:           {status(bucket_found)}")
print(f"Invocation logging:  {status(logging_found)}")

In [ ]:
DELETE_ROLES_AND_LOGS = False  # Set to True only in your own account, not at an AWS event.

ROLE_NAMES = ["aim317-bedrock-logging-role", "aim317-knowledge-base-role", "aim317-lambda-role"]

if DELETE_ROLES_AND_LOGS:
    for role_name in ROLE_NAMES:
        try:
            for policy_name in iam.list_role_policies(RoleName=role_name)["PolicyNames"]:
                iam.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
            for policy in iam.list_attached_role_policies(RoleName=role_name)["AttachedPolicies"]:
                iam.detach_role_policy(RoleName=role_name, PolicyArn=policy["PolicyArn"])
            iam.delete_role(RoleName=role_name)
            print(f"Deleted role {role_name}")
        except iam.exceptions.NoSuchEntityException:
            print(f"Role {role_name} not found")
    try:
        logs.delete_log_group(logGroupName=LOG_GROUP_NAME)
        print(f"Deleted log group {LOG_GROUP_NAME}")
    except logs.exceptions.ResourceNotFoundException:
        print(f"Log group {LOG_GROUP_NAME} not found")
else:
    print("Skipped. Set DELETE_ROLES_AND_LOGS = True to delete the IAM roles and the log group.")

## 9. Conclusion and key takeaways <a id="conclusion"></a>

- Knowledge bases, guardrails, Lambda functions and S3 buckets remain in your account until you delete them, and some incur charges while they exist.
- Delete a knowledge base's data sources before the knowledge base. A data source with the `DELETE` data deletion policy also removes its indexed data.
- Model invocation logging applies to the whole account and Region. Turn it off, or point it to a log group you intend to keep, when the workshop ends.
- Resources that CloudFormation manages, such as the Lambda function at AWS events, are reset here and deleted with their stack.

## 10. Next steps <a id="next-steps"></a>

Some directions to explore from here:

- **Batch inference**: check thousands of expense reports at a lower price when nobody is waiting for the answer. See [batch inference](https://docs.aws.amazon.com/bedrock/latest/userguide/batch-inference.html).
- **Tool use and agents**: let the model decide which steps to take. The [AgentCore stretch notebook](stretch_agentcore.ipynb) turns this assistant into an agent.
- **More Knowledge Base connectors**: connect SharePoint, Confluence or a web crawler instead of S3. See [supported data sources](https://docs.aws.amazon.com/bedrock/latest/userguide/data-source-connectors.html).